# v1 — Augmentation + Normalize + OneCycleLR

**변경사항 (from baseline_code_v0)**
- Normalize 추가 (ImageNet mean/std)
- Data Augmentation 추가 (RandomFlip, RandomRotation, ColorJitter)
- OneCycleLR scheduler 추가 (batch마다 호출)
- 데이터 경로 로컬 환경 지원 (`Data/fruit_data_raw`)
- `n_epochs = 1` (과제 제약)
- `if __name__ == '__main__'` 제거 (notebook은 불필요)
- num_workers=2

**Submission**: `submission_v1_augment_normalize_onecyclelr.csv`

## 1. Imports

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## 2. Reproducibility

In [ ]:
myseed = 6666
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(myseed)
np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

print(f"Seed fixed: {myseed}")

## 3. Data Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Validation / Test: resize + normalize only
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Train: augmentation + normalize
train_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")

## 4. Dataset

In [ ]:
class FruitDataset(Dataset):
    """Train/valid: walks class subfolders. Test: flat folder of anonymized jpgs."""

    def __init__(self, root, tfm=test_tfm, class_to_idx=None, is_test=False):
        super().__init__()
        self.root = Path(root)
        self.transform = tfm
        self.is_test = is_test

        if is_test:
            self.samples = sorted(self.root.glob("*.jpg"))
            self.labels = None
        else:
            assert class_to_idx is not None, "class_to_idx is required for train/valid"
            self.class_to_idx = class_to_idx
            samples, labels = [], []
            for class_dir in sorted(self.root.iterdir()):
                if not class_dir.is_dir():
                    continue
                if class_dir.name not in class_to_idx:
                    raise ValueError(f"Unknown class folder: {class_dir.name}")
                label = class_to_idx[class_dir.name]
                for img in sorted(class_dir.glob("*.jpg")):
                    samples.append(img)
                    labels.append(label)
            self.samples = samples
            self.labels = labels

        print(f"Loaded {len(self.samples)} samples from {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname = self.samples[idx]
        im = Image.open(fname).convert("RGB")
        im = self.transform(im)
        if self.is_test:
            return im, -1, fname.stem
        return im, self.labels[idx]

## 5. Model

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_classes):
        super(Classifier, self).__init__()
        # input dimension [3, 128, 128]
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),    # [64, 128, 128]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),         # [64, 64, 64]

            nn.Conv2d(64, 128, 3, 1, 1),   # [128, 64, 64]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),         # [128, 32, 32]

            nn.Conv2d(128, 256, 3, 1, 1),  # [256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),         # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1),  # [512, 16, 16]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),         # [512, 8, 8]

            nn.Conv2d(512, 512, 3, 1, 1),  # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),         # [512, 4, 4]
        )
        self.fc = nn.Sequential(
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size(0), -1)
        return self.fc(out)


print("Model defined.")

## 6. Dataset Path & Class Mapping

In [ ]:
def resolve_dataset_dir():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for child in kaggle_input.iterdir():
            if child.is_dir() and (child / "classes.txt").is_file():
                return child
    local_path = Path("./Data/fruit_data_raw")
    if local_path.is_dir():
        return local_path
    return Path("./data")


_dataset_dir = resolve_dataset_dir()
print(f"Dataset directory: {_dataset_dir}")

with open(_dataset_dir / "classes.txt", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)
print(f"Number of classes: {num_classes}")

## 7. Data Loaders

In [ ]:
batch_size = 64

train_set    = FruitDataset(_dataset_dir / "train", tfm=train_tfm, class_to_idx=class_to_idx)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

valid_set    = FruitDataset(_dataset_dir / "valid", tfm=test_tfm, class_to_idx=class_to_idx)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

## 8. Training Setup

In [ ]:
device   = "cuda" if torch.cuda.is_available() else "cpu"
n_epochs = 1   # assignment constraint: 1 epoch only

model     = Classifier(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

# OneCycleLR: ramps LR up then down over all batches in 1 epoch
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.01,
    steps_per_epoch=len(train_loader),
    epochs=n_epochs,
)

best_acc   = 0
_exp_name  = "fruit_classification_v1"

print(f"Device   : {device}")
print(f"Epochs   : {n_epochs}")
print(f"Batches  : {len(train_loader)}")

## 9. Training & Validation

In [ ]:
history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}

for epoch in range(n_epochs):

    # ---------- Training ----------
    model.train()
    train_loss, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"[Train {epoch+1}/{n_epochs}]"):
        imgs, labels = batch

        optimizer.zero_grad()
        logits = model(imgs.to(device))
        loss   = criterion(logits, labels.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        optimizer.step()
        scheduler.step()

        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        train_loss.append(loss.item())
        train_accs.append(acc.item())

    t_loss = sum(train_loss) / len(train_loss)
    t_acc  = sum(train_accs) / len(train_accs)
    print(f"[ Train | {epoch+1:03d}/{n_epochs:03d} ] loss = {t_loss:.5f}, acc = {t_acc:.5f}")

    # ---------- Validation ----------
    model.eval()
    valid_loss, valid_accs = [], []

    for batch in tqdm(valid_loader, desc=f"[Valid {epoch+1}/{n_epochs}]"):
        imgs, labels = batch
        with torch.no_grad():
            logits = model(imgs.to(device))
        loss = criterion(logits, labels.to(device))
        acc  = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        valid_loss.append(loss.item())
        valid_accs.append(acc.item())

    v_loss = sum(valid_loss) / len(valid_loss)
    v_acc  = sum(valid_accs) / len(valid_accs)
    tag    = " -> best" if v_acc > best_acc else ""
    print(f"[ Valid | {epoch+1:03d}/{n_epochs:03d} ] loss = {v_loss:.5f}, acc = {v_acc:.5f}{tag}")

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["valid_loss"].append(v_loss)
    history["valid_acc"].append(v_acc)

    if v_acc > best_acc:
        print(f"Best model found at epoch {epoch+1}, saving...")
        torch.save(model.state_dict(), f"{_exp_name}_best.ckpt")
        best_acc = v_acc

print(f"\nTraining done. Best valid acc: {best_acc:.5f}")

## 10. 결과 시각화

In [ ]:
epochs_range = range(1, n_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_range, history["valid_loss"], label="Valid Loss", marker="o")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(epochs_range, history["valid_acc"], label="Valid Acc", marker="o")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("v1 — Augment + Normalize + OneCycleLR", fontsize=13)
plt.tight_layout()
plt.savefig("v1_training_curve.png", dpi=150)
plt.show()
print(f"Final — Train acc: {history['train_acc'][-1]:.4f} | Valid acc: {history['valid_acc'][-1]:.4f}")

## 11. Testing

In [ ]:
test_set    = FruitDataset(_dataset_dir / "test", tfm=test_tfm, is_test=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

model_best = Classifier(num_classes).to(device)
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt"))
model_best.eval()

prediction, file_ids = [], []
with torch.no_grad():
    for data, _, file_id in tqdm(test_loader, desc="Testing"):
        test_pred  = model_best(data.to(device))
        test_label = np.argmax(test_pred.cpu().data.numpy(), axis=1)
        prediction += test_label.tolist()
        file_ids   += list(file_id)

print(f"Predictions: {len(prediction)}")

## 12. Submission

In [ ]:
submission_dir  = Path("./Data/submissions")
submission_dir.mkdir(parents=True, exist_ok=True)
submission_name = submission_dir / "submission_v1_augment_normalize_onecyclelr.csv"

df = pd.DataFrame({"ID": file_ids, "Category": prediction})
df.to_csv(submission_name, index=False)

print(f"Saved: {submission_name}")
df.head(10)